# Standard Pre-Training (SPT) | OmniMATH | Qwen2.5-1.5B | L4

Pre-trains Qwen2.5-1.5B on OmniMATH using standard cross-entropy (NTP).
Produces the SPT checkpoint used in spt_grpo.ipynb and spt_intuitor.ipynb.

Result: NTP Accuracy 26.5% | Perplexity 2.47

> Selecione **L4** antes de rodar.

## Celula 0 -- GPU

In [ ]:
import subprocess, torch
print(subprocess.run(['nvidia-smi'], capture_output=True, text=True).stdout)
if torch.cuda.is_available():
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'GPU : {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {vram:.1f} GB')
    assert vram >= 20
    print('OK GPU verificada')

Thu Jun 11 14:56:35 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA L4                      Off |   00000000:00:03.0 Off |                    0 |
| N/A   73C    P0             33W /   72W |    8806MiB /  23034MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## Celula 1 -- Instalacao

In [ ]:
!pip install -q transformers==4.51.0 datasets==3.2.0 accelerate==1.4.0 peft==0.15.1
!pip install -q bitsandbytes sentencepiece
print('OK')

OK


## Celula 2 -- Config

In [ ]:
import os, warnings, logging, math
import numpy as np
import torch
from transformers import (
    AutoModelForCausalLM, AutoTokenizer,
    TrainingArguments, Trainer, DataCollatorForLanguageModeling,
    set_seed,
)
from peft import LoraConfig, TaskType, get_peft_model
from datasets import load_dataset
warnings.filterwarnings('ignore')
logging.getLogger('transformers').setLevel(logging.WARNING)

MODEL_NAME    = 'Qwen/Qwen2.5-1.5B'
N_TRAIN       = 1000
N_EVAL        = 100
MAX_LENGTH    = 512
SEED          = 42
LEARNING_RATE = 3e-4
LORA_R        = 16
LORA_ALPHA    = 32
NUM_EPOCHS    = 1
BATCH_SIZE    = 4
GRAD_ACCUM    = 4
OUTPUT_DIR    = 'qwen-1.5b-spt-omnimath'

print(f'Modelo    : {MODEL_NAME}')
print(f'N_TRAIN   : {N_TRAIN}')
print(f'Output    : {OUTPUT_DIR}')
set_seed(SEED)

Modelo    : Qwen/Qwen2.5-1.5B
N_TRAIN   : 1000
Output    : qwen-1.5b-spt-omnimath


## Celula 3 -- Google Drive

In [ ]:
SAVE_TO_DRIVE = True
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    SAVE_DIR = f'/content/drive/MyDrive/intuitor-ic/{OUTPUT_DIR}'
    os.makedirs(SAVE_DIR, exist_ok=True)
    print(f'OK Checkpoints em: {SAVE_DIR}')
else:
    SAVE_DIR = f'/content/{OUTPUT_DIR}'
    os.makedirs(SAVE_DIR, exist_ok=True)
    print(f'Salvando localmente: {SAVE_DIR}')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
OK Checkpoints em: /content/drive/MyDrive/intuitor-ic/qwen-1.5b-spt-omnimath


## Celula 4 -- Dataset OmniMATH

Formata cada problema como texto continuo:
```
Problem: {enunciado}
Solution: {solucao}
```

In [ ]:
print('Carregando OmniMATH...')
raw = load_dataset('KbsdJames/Omni-MATH', split='test', trust_remote_code=True)
raw = raw.shuffle(seed=SEED)
train_raw = raw.select(range(N_TRAIN))
eval_raw  = raw.select(range(N_TRAIN, N_TRAIN + N_EVAL))
print(f'OK Treino : {len(train_raw)} problemas')
print(f'OK Eval   : {len(eval_raw)} problemas')
print('\nExemplo:')
print(train_raw[0]['problem'][:200], '...')

Carregando OmniMATH...
OK Treino : 1000 problemas
OK Eval   : 100 problemas

Exemplo:
Choose positive integers $b_1, b_2, \dotsc$ satisfying
\[1=\frac{b_1}{1^2} > \frac{b_2}{2^2} > \frac{b_3}{3^2} > \frac{b_4}{4^2} > \dotsb\]
and let $r$ denote the largest real number satisfying $\tfra ...


## Celula 5 -- Tokenizer e formatacao

In [ ]:
print(f'Carregando tokenizer: {MODEL_NAME}')
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
tokenizer.padding_side = 'right'
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

def format_and_tokenize(example):
    problem  = example.get('problem', '')
    solution = example.get('solution', '')
    text = f'Problem: {problem}\nSolution: {solution}'
    tokens = tokenizer(
        text, truncation=True, max_length=MAX_LENGTH,
        padding='max_length', return_tensors=None,
    )
    labels = tokens['input_ids'].copy()
    for i, mask in enumerate(tokens['attention_mask']):
        if mask == 0:
            labels[i] = -100
    tokens['labels'] = labels
    return tokens

print('Tokenizando treino...')
train_dataset = train_raw.map(format_and_tokenize, batched=False, remove_columns=train_raw.column_names)
print('Tokenizando eval...')
eval_dataset = eval_raw.map(format_and_tokenize, batched=False, remove_columns=eval_raw.column_names)

lengths = [sum(x) for x in train_dataset['attention_mask']]
print(f'OK {len(train_dataset)} exemplos | comprimento medio: {int(np.mean(lengths))} tokens')

Carregando tokenizer: Qwen/Qwen2.5-1.5B
Tokenizando treino...
Tokenizando eval...
OK 1000 exemplos | comprimento medio: 371 tokens


## Celula 6 -- Modelo com LoRA

In [ ]:
print(f'Carregando modelo: {MODEL_NAME}')
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, torch_dtype=torch.bfloat16,
    device_map='auto', trust_remote_code=True,
)
model.config.pad_token_id = tokenizer.pad_token_id

lora_config = LoraConfig(
    r=LORA_R, lora_alpha=LORA_ALPHA,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj',
                    'up_proj', 'down_proj', 'gate_proj'],
    task_type=TaskType.CAUSAL_LM, lora_dropout=0.05, bias='none',
)
model = get_peft_model(model, lora_config)
model.enable_input_require_grads()

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
vram_gb   = torch.cuda.memory_allocated() / 1e9
print(f'OK LoRA | treinaveis: {trainable:,} ({100*trainable/total:.2f}%) | VRAM: {vram_gb:.2f} GB')

Carregando modelo: Qwen/Qwen2.5-1.5B
OK LoRA | treinaveis: 18,464,768 (1.18%) | VRAM: 6.49 GB


## Celula 7 -- TrainingArguments e Trainer

SPT usa o `Trainer` padrao com cross-entropy -- sem GRPO, sem reward.

In [ ]:
training_args = TrainingArguments(
    output_dir=SAVE_DIR,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LEARNING_RATE,
    lr_scheduler_type='cosine',
    warmup_ratio=0.1,
    weight_decay=0.01,
    bf16=True, fp16=False,
    gradient_checkpointing=True,
    logging_steps=10,
    eval_strategy='epoch',
    save_strategy='epoch',
    save_total_limit=1,
    report_to='none',
    seed=SEED,
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',
    greater_is_better=False,
)

data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

trainer = Trainer(
    model=model, args=training_args,
    train_dataset=train_dataset, eval_dataset=eval_dataset,
    data_collator=data_collator, processing_class=tokenizer,
)

steps = len(train_dataset) // (BATCH_SIZE * GRAD_ACCUM)
print(f'OK Trainer | batch efetivo: {BATCH_SIZE * GRAD_ACCUM} | steps/epoca: ~{steps}')
print('Tempo estimado: ~30-45 min na L4')

No label_names provided for model class `PeftModelForCausalLM`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.


OK Trainer | batch efetivo: 16 | steps/epoca: ~62
Tempo estimado: ~30-45 min na L4


In [ ]:
import math

def evaluate_perplexity(model, eval_dataset, tokenizer, batch_size=4, desc=''):
    """Calcula perplexidade no eval_dataset.

    Passa os exemplos ja tokenizados pelo modelo e acumula a cross-entropy loss.
    Perplexidade = exp(loss_media) -- metrica padrao de pre-treinamento.
    """
    from torch.utils.data import DataLoader
    import torch

    model.eval()
    total_loss   = 0.0
    total_tokens = 0

    loader = DataLoader(
        eval_dataset,
        batch_size=batch_size,
        shuffle=False,
        collate_fn=lambda x: {
            k: torch.tensor([d[k] for d in x]).to(model.device)
            for k in ['input_ids', 'attention_mask', 'labels']
        },
    )

    print(f'\n{"="*60}')
    print(f'AVALIACAO PERPLEXIDADE ({len(eval_dataset)} exemplos) {desc}')
    print(f'{"="*60}')

    with torch.no_grad():
        for step, batch in enumerate(loader):
            outputs = model(
                input_ids=batch['input_ids'],
                attention_mask=batch['attention_mask'],
                labels=batch['labels'],
            )
            loss = outputs.loss

            # Conta apenas tokens validos (ignora padding -100)
            n_valid = (batch['labels'] != -100).sum().item()
            total_loss   += loss.item() * n_valid
            total_tokens += n_valid

            if (step + 1) % 5 == 0:
                partial_ppl = math.exp(total_loss / total_tokens)
                print(f'  [{(step+1)*batch_size}/{len(eval_dataset)}] perplexidade parcial: {partial_ppl:.2f}')

    avg_loss = total_loss / total_tokens
    ppl      = math.exp(avg_loss)

    print(f'\n{"="*60}')
    print(f'RESULTADO -- loss: {avg_loss:.4f} | perplexidade: {ppl:.2f}')
    print(f'{"="*60}')
    return avg_loss, ppl


print('Avaliando modelo BASE no OmniMATH (antes do SPT)...')
base_loss, base_ppl = evaluate_perplexity(
    model, eval_dataset, tokenizer,
    batch_size=4,
    desc='-- BASE MODEL (0 epocas SPT)'
)
print(f'\nOK BASE SCORE registrado:')
print(f'   Loss        : {base_loss:.4f}')
print(f'   Perplexidade: {base_ppl:.2f}')
print('Prossiga para a Celula 8 para iniciar o treino SPT.')

Avaliando modelo BASE no OmniMATH (antes do SPT)...

AVALIACAO PERPLEXIDADE (100 exemplos) -- BASE MODEL (0 epocas SPT)
  [20/100] perplexidade parcial: 2.64
  [40/100] perplexidade parcial: 2.76
  [60/100] perplexidade parcial: 2.67
  [80/100] perplexidade parcial: 2.61
  [100/100] perplexidade parcial: 2.65

RESULTADO -- loss: 0.9756 | perplexidade: 2.65

OK BASE SCORE registrado:
   Loss        : 0.9756
   Perplexidade: 2.65
Prossiga para a Celula 8 para iniciar o treino SPT.


In [ ]:
import json, torch, os

N_NTP_EVAL = 200   # exemplos para avaliacao NTP

# Caminho do rpt_dataset.jsonl gerado pelo rpt_preprocessing.ipynb
NTP_DATASET_PATH = '/content/drive/MyDrive/intuitor-ic/rpt_dataset.jsonl'

def load_ntp_eval(path, n=N_NTP_EVAL, seed=42):
    """Carrega N exemplos do rpt_dataset.jsonl para avaliacao NTP.
    Usa os ultimos exemplos do arquivo para evitar overlap com o treino RPT.
    """
    if not os.path.exists(path):
        print(f'Dataset NTP nao encontrado: {path}')
        print('Rode rpt_preprocessing.ipynb para gerar o dataset.')
        return None
    records = []
    with open(path) as f:
        for line in f:
            records.append(json.loads(line))
    # Pega os ultimos N exemplos -- sem overlap com treino RPT (que usa os primeiros)
    eval_records = records[-n:]
    print(f'OK {len(eval_records)} exemplos NTP carregados (ultimos do arquivo)')
    return eval_records


def evaluate_ntp_accuracy(model, tokenizer, eval_records, desc=''):
    """Acuracia de predicao de proximo token (metrica principal do paper RPT).

    Para cada posicao de alta entropia:
      - Passa o contexto pelo modelo
      - Verifica se argmax(logits) == target_token_id
    Sem chain-of-thought -- predicao direta, greedy.
    """
    model.eval()
    correct = 0
    total   = len(eval_records)

    print(f'\n{"="*60}')
    print(f'AVALIACAO NTP ACCURACY ({total} exemplos) {desc}')
    print(f'{"="*60}')

    for i, record in enumerate(eval_records):
        context_str = record['context']
        target_id   = record['target_token_id']

        inputs = tokenizer(
            context_str, return_tensors='pt',
            truncation=True, max_length=512
        ).to(model.device)

        with torch.no_grad():
            outputs      = model(**inputs)
            last_logits  = outputs.logits[0, -1, :]      # ultima posicao
            predicted_id = int(last_logits.argmax().item())

        if predicted_id == target_id:
            correct += 1

        if (i + 1) % 50 == 0:
            print(f'  [{i+1}/{total}] acuracia parcial: {correct/(i+1):.4f} ({correct/(i+1)*100:.2f}%)')

    accuracy = correct / total
    print(f'\n{"="*60}')
    print(f'RESULTADO NTP -- acuracia: {accuracy:.4f} ({accuracy*100:.2f}%)')
    print(f'Corretos: {correct}/{total}')
    print(f'{"="*60}')
    return accuracy


# Carrega dataset de avaliacao NTP
ntp_eval_records = load_ntp_eval(NTP_DATASET_PATH, n=N_NTP_EVAL)

if ntp_eval_records:
    base_ntp_score = evaluate_ntp_accuracy(
        model, tokenizer, ntp_eval_records,
        desc='-- BASE MODEL (0 epocas SPT)'
    )
    print(f'\nOK BASE NTP SCORE registrado: {base_ntp_score*100:.2f}%')
else:
    base_ntp_score = None
    print('Avaliacao NTP pulada -- dataset nao encontrado.')
    print('Voce pode rodar esta celula depois de gerar o rpt_dataset.jsonl')

OK 200 exemplos NTP carregados (ultimos do arquivo)

AVALIACAO NTP ACCURACY (200 exemplos) -- BASE MODEL (0 epocas SPT)
  [50/200] acuracia parcial: 0.1800 (18.00%)
  [100/200] acuracia parcial: 0.2500 (25.00%)
  [150/200] acuracia parcial: 0.2333 (23.33%)
  [200/200] acuracia parcial: 0.2350 (23.50%)

RESULTADO NTP -- acuracia: 0.2350 (23.50%)
Corretos: 47/200

OK BASE NTP SCORE registrado: 23.50%


## Celula 8 -- Treino SPT

In [ ]:
print('='*60)
print(f'TREINO SPT | {MODEL_NAME}')
print(f'Dataset: OmniMATH ({N_TRAIN} problemas)')
print(f'Objetivo: NTP padrao (cross-entropy)')
print('='*60)

train_result = trainer.train()

train_loss = train_result.training_loss
train_ppl  = math.exp(train_loss)
print(f'OK Treino concluido!')
print(f'   Train loss : {train_loss:.4f} (ppl={train_ppl:.2f})')

eval_results = trainer.evaluate()
eval_loss    = eval_results['eval_loss']
eval_ppl     = math.exp(eval_loss)
print(f'   Eval loss  : {eval_loss:.4f} (ppl={eval_ppl:.2f})')

trainer.save_model(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)
print(f'Modelo SPT salvo em: {SAVE_DIR}')

TREINO SPT | Qwen/Qwen2.5-1.5B
Dataset: OmniMATH (1000 problemas)
Objetivo: NTP padrao (cross-entropy)


Epoch,Training Loss,Validation Loss
0,0.943000,0.903837


OK Treino concluido!
   Train loss : 0.9573 (ppl=2.60)


   Eval loss  : 0.9038 (ppl=2.47)
Modelo SPT salvo em: /content/drive/MyDrive/intuitor-ic/qwen-1.5b-spt-omnimath


## Celula 9 -- Avaliacao qualitativa

In [ ]:
model.eval()
print('Exemplos de geracao (SPT pos-treinamento):\n')

test_prompts = [
    'Problem: Find all integers n such that',
    'Problem: Prove that for all positive real numbers',
    'Problem: Let f be a continuous function such that',
]

for prompt in test_prompts:
    inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
    with torch.no_grad():
        out = model.generate(
            **inputs, max_new_tokens=80, do_sample=True,
            temperature=0.8, top_p=0.95,
            pad_token_id=tokenizer.pad_token_id,
        )
    completion = tokenizer.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
    print(f'Prompt  : {prompt}')
    print(f'Geracao : {completion[:150]}')
    print('-'*60)

Exemplos de geracao (SPT pos-treinamento):

Prompt  : Problem: Find all integers n such that
Geracao :  $n^{2} \mid n^{3}+n+1$.
Solution: Note that $n^{3}+n+1=(n+1)(n^{2}-n+1)$. If $n=0$, then $0 \mid 0+0+1$, so $n=0$ is a solution. If $n \neq 0
------------------------------------------------------------
Prompt  : Problem: Prove that for all positive real numbers
Geracao :  $a, b, c$ with $a + b + c = 1$, the following inequality holds:

$$\sqrt{\frac{a}{b + c}} + \sqrt{\frac{b}{c + a}} + \sqrt{\frac{c}{a + b}} \geq \sqr
------------------------------------------------------------
Prompt  : Problem: Let f be a continuous function such that
Geracao :  $\int_{0}^{1} f(x) \, \mathrm{d}x = 0$ and $f(x) > 0$ for all $x \in (0,1)$. Show that $\left(\int_{0}^{1} x^{2} f(x) \, \mathrm{d}x\right)^{2} > \in
------------------------------------------------------------


## Celula 10 -- Resumo e proximos passos

In [ ]:
print('='*60)
print('RESUMO SPT')
print('='*60)
print(f'Modelo    : {MODEL_NAME}')
print(f'Dataset   : OmniMATH ({N_TRAIN} problemas)')
print(f'Objetivo  : NTP padrao (cross-entropy)')
print(f'LoRA r    : {LORA_R}')
print(f'Checkpoint: {SAVE_DIR}')
print(f'Train loss: {train_loss:.4f} (ppl={train_ppl:.2f})')
print(f'Eval  loss: {eval_loss:.4f} (ppl={eval_ppl:.2f})')
print('='*60)


RESUMO SPT
Modelo    : Qwen/Qwen2.5-1.5B
Dataset   : OmniMATH (1000 problemas)
Objetivo  : NTP padrao (cross-entropy)
LoRA r    : 16
Checkpoint: /content/drive/MyDrive/intuitor-ic/qwen-1.5b-spt-omnimath
Train loss: 0.9573 (ppl=2.60)
Eval  loss: 0.9038 (ppl=2.47)

Proximos passos:
  1. rpt_preprocessing.ipynb  -- filtragem por entropia
  2. rpt_training.ipynb       -- pre-treinamento com RL (RPT)
  3. Fine-tuning SPT -> GRPO  e SPT -> Intuitor
  4. Fine-tuning RPT -> GRPO  e RPT -> Intuitor

Tabela alvo da IC:
  BASE -> GRPO     : 69.0%  (ja medido)
  BASE -> Intuitor : 31.5%  (ja medido)
  SPT  -> GRPO     : ?
  SPT  -> Intuitor : ?
  RPT  -> GRPO     : ?
  RPT  -> Intuitor : ?


In [ ]:
if ntp_eval_records:
    spt_ntp_score = evaluate_ntp_accuracy(
        model, tokenizer, ntp_eval_records,
        desc='-- SPT (pos-treinamento)'
    )

    print('\n' + '='*60)
    print('COMPARACAO NTP -- BASE vs SPT')
    print('='*60)
    if base_ntp_score is not None:
        delta = (spt_ntp_score - base_ntp_score) * 100
        print(f'  BASE (0 epocas) : {base_ntp_score:.4f} ({base_ntp_score*100:.2f}%)')
        print(f'  SPT pos-treino  : {spt_ntp_score:.4f} ({spt_ntp_score*100:.2f}%)')
        print(f'  Delta           : {delta:+.2f} p.p.')
        if delta > 0:
            print('  -> SPT melhorou a acuracia NTP no OmniMATH')
        else:
            print('  -> SPT nao melhorou a acuracia NTP (esperado -- SPT aprende distribuicao, nao acuracia pontual)')
    print('='*60)
    print()
    print('Referencia do paper RPT (Tabela 1, Secao 4.1):')
    print('  R1-Distill-Qwen-14B (NTP padrao) : ~30-40% (tokens medios/dificeis)')
    print('  RPT-14B (apos RPT)               : ~33-46% (melhora consistente)')
    print()
    print('Nota: seus resultados sao com Qwen2.5-1.5B (modelo menor) --')
    print('      valores absolutos menores sao esperados, delta relativo importa mais.')
else:
    spt_ntp_score = None
    print('Avaliacao NTP pulada -- dataset nao encontrado.')


AVALIACAO NTP ACCURACY (200 exemplos) -- SPT (pos-treinamento)
  [50/200] acuracia parcial: 0.2600 (26.00%)
  [100/200] acuracia parcial: 0.3000 (30.00%)
  [150/200] acuracia parcial: 0.2667 (26.67%)
  [200/200] acuracia parcial: 0.2700 (27.00%)

RESULTADO NTP -- acuracia: 0.2700 (27.00%)
Corretos: 54/200

COMPARACAO NTP -- BASE vs SPT
  BASE (0 epocas) : 0.2350 (23.50%)
  SPT pos-treino  : 0.2700 (27.00%)
  Delta           : +3.50 p.p.
  -> SPT melhorou a acuracia NTP no OmniMATH

Referencia do paper RPT (Tabela 1, Secao 4.1):
  R1-Distill-Qwen-14B (NTP padrao) : ~30-40% (tokens medios/dificeis)
  RPT-14B (apos RPT)               : ~33-46% (melhora consistente)

Nota: seus resultados sao com Qwen2.5-1.5B (modelo menor) --
      valores absolutos menores sao esperados, delta relativo importa mais.
